In [5]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ==========================================
# 1. LOAD DATASET
# ==========================================
iris = load_iris()
X = iris.data
y = iris.target.reshape(-1, 1)

# One-hot encoding
encoder = OneHotEncoder(sparse_output=False)
y_encoded = encoder.fit_transform(y)

# Split data (sama untuk semua percobaan)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y.ravel()
)

# Normalisasi data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ==========================================
# 2. FUNGSI MEMBANGUN DAN MELATIH MODEL
# ==========================================
def train_model(name, neurons, activation):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)

    model = tf.keras.Sequential([
        tf.keras.Input(shape=(4,)),
        tf.keras.layers.Dense(
            neurons[0],
            activation=activation
        ),
        tf.keras.layers.Dense(
            neurons[1],
            activation=activation
        ),
        tf.keras.layers.Dense(3, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    print(f"\n=== {name} ===")
    print(f"Neuron hidden layer: {neurons}")
    print(f"Aktivasi: {activation}")

    model.fit(
        X_train,
        y_train,
        epochs=50,
        batch_size=8,
        verbose=0
    )

    loss, accuracy = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )

    print(f"Test Loss     : {loss:.4f}")
    print(f"Test Accuracy : {accuracy:.4f}")

    return {
        "Percobaan": name,
        "Neuron Hidden Layer": str(neurons),
        "Aktivasi": activation,
        "Loss": loss,
        "Akurasi": accuracy
    }


# ==========================================
# 3. TUGAS 2: PERUBAHAN JUMLAH NEURON
# ==========================================
results = []

# Konfigurasi awal: 10 dan 8 neuron
results.append(
    train_model(
        "Konfigurasi Awal",
        [10, 8],
        "relu"
    )
)

# Percobaan: 16 dan 12 neuron
results.append(
    train_model(
        "Hidden Layer Lebih Besar",
        [16, 12],
        "relu"
    )
)


# ==========================================
# 4. TUGAS 3: SIGMOID VS RELU
# ==========================================

# Sigmoid: 10 dan 8 neuron
results.append(
    train_model(
        "Aktivasi Sigmoid",
        [10, 8],
        "sigmoid"
    )
)

# ReLU: 10 dan 8 neuron
results.append(
    train_model(
        "Aktivasi ReLU",
        [10, 8],
        "relu"
    )
)


# ==========================================
# 5. PERBANDINGAN HASIL
# ==========================================
df = pd.DataFrame(results)

print("\n=== TABEL PERBANDINGAN HASIL ===")
print(df.to_string(index=False))

print("\n=== AKURASI DALAM PERSEN ===")
for _, row in df.iterrows():
    print(
        f"{row['Percobaan']}: "
        f"Loss = {row['Loss']:.4f}, "
        f"Akurasi = {row['Akurasi'] * 100:.2f}%"
    )



=== Konfigurasi Awal ===
Neuron hidden layer: [10, 8]
Aktivasi: relu
Test Loss     : 0.1783
Test Accuracy : 0.9333

=== Hidden Layer Lebih Besar ===
Neuron hidden layer: [16, 12]
Aktivasi: relu
Test Loss     : 0.2071
Test Accuracy : 0.9000

=== Aktivasi Sigmoid ===
Neuron hidden layer: [10, 8]
Aktivasi: sigmoid
Test Loss     : 0.5177
Test Accuracy : 0.8000

=== Aktivasi ReLU ===
Neuron hidden layer: [10, 8]
Aktivasi: relu
Test Loss     : 0.1783
Test Accuracy : 0.9333

=== TABEL PERBANDINGAN HASIL ===
               Percobaan Neuron Hidden Layer Aktivasi     Loss  Akurasi
        Konfigurasi Awal             [10, 8]     relu 0.178294 0.933333
Hidden Layer Lebih Besar            [16, 12]     relu 0.207146 0.900000
        Aktivasi Sigmoid             [10, 8]  sigmoid 0.517665 0.800000
           Aktivasi ReLU             [10, 8]     relu 0.178294 0.933333

=== AKURASI DALAM PERSEN ===
Konfigurasi Awal: Loss = 0.1783, Akurasi = 93.33%
Hidden Layer Lebih Besar: Loss = 0.2071, Akurasi = 9